<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

Э
В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [27]:
!pip install -q datasets

Импорт необходимых библиотек

In [28]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [29]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [30]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [31]:
# Загрузим датасет
dataset = load_dataset('imdb')

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [32]:
sentences = []
word_threshold = 32

for text in tqdm(dataset['train']['text']):
    for sent in sent_tokenize(text):
        tokens = word_tokenize(sent)
        if len(tokens) < word_threshold:
            sentences.append(sent)

  0%|          | 0/25000 [00:00<?, ?it/s]

In [33]:
print("Всего предложений:", len(sentences))

Всего предложений: 198801


Посчитаем для каждого слова его встречаемость.

In [34]:
words = Counter()

for sent in tqdm(sentences):
    tokens = word_tokenize(sent.lower())
    words.update(tokens)

print("Уникальных слов встретилось:", len(words))

  0%|          | 0/198801 [00:00<?, ?it/s]

Уникальных слов встретилось: 68415


Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [35]:
vocab = set()
vocab_size = 40000
most_common_words = [word for word, _ in words.most_common(vocab_size)]

special_tokens = {'<pad>', '<unk>', '<bos>', '<eos>'}
vocab = set(most_common_words) | special_tokens

In [36]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [37]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [38]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [39]:
class WordDataset(Dataset):
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        sentence = self.data[idx]
        # tokens = word_tokenize(sentence)
        tokens = word_tokenize(sentence.lower())

        tokenized_sentence = [self.bos_id]
        tokenized_sentence += [word2ind.get(token, self.unk_id) for token in tokens]
        tokenized_sentence += [self.eos_id]

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [40]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [50]:
train_sentences, temp_sentences = train_test_split(sentences, test_size=0.2, random_state=42)
eval_sentences, test_sentences = train_test_split(temp_sentences, test_size=0.5, random_state=42)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size, shuffle=True)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size, shuffle=False)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size, shuffle=False)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [46]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids'])
            logits = logits.view(-1, logits.shape[-1])
            loss = criterion(logits, batch['target_ids'].flatten())
            total_loss += loss.item()
            total_batches += 1
    mean_losses = total_loss / total_batches
    perplexity = np.exp(mean_losses)

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [43]:
def train_model(model, optimizer, criterion, train_dataloader, eval_dataloader, num_epochs):
  train_losses = []
  eval_perplexities = []

  for epoch in range(num_epochs):
    model.train()
    epoch_losses = []
    for batch in train_dataloader:
      optimizer.zero_grad()
      logits = model(batch['input_ids'])
      logits = logits.view(-1, logits.shape[-1])
      loss = criterion(logits, batch['target_ids'].flatten())

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      optimizer.step()

      epoch_losses.append(loss.item())
    average_loss = sum(epoch_losses) / len(epoch_losses)
    train_losses.append(average_loss)
    eval_perplexity = evaluate(model, criterion, eval_dataloader)
    eval_perplexities.append(eval_perplexity)
    print(f'{epoch+1}/{num_epochs}, train_loss={average_loss:.4f}, eval_perplexity={eval_perplexity:.4f}')
  return train_losses, eval_perplexities

### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [49]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=1, dropout=0.0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind["<pad>"])
        self.rnn = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.linear = nn.Linear(hidden_dim, vocab_size)
    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        output, _ = self.rnn(embeddings)
        logits = self.linear(output)

        return logits

In [51]:
model1 = LanguageModel(
    vocab_size=len(word2ind),
    embedding_dim=128,
    hidden_dim=256,
    num_layers=1,
    dropout=0.0
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-3)

train_losses1, eval_perplexities1 = train_model(
    model=model1,
    optimizer=optimizer,
    criterion=criterion,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    num_epochs=5
)

1/5, train_loss=5.3471, eval_perplexity=137.9381
2/5, train_loss=4.7128, eval_perplexity=114.1537
3/5, train_loss=4.4667, eval_perplexity=106.5521
4/5, train_loss=4.2931, eval_perplexity=104.1914
5/5, train_loss=4.1557, eval_perplexity=104.3621


In [52]:
print("Final perplexity:", eval_perplexities1[-1])

Final perplexity: 104.36214397383723


### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [62]:
class LanguageModel_2(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=3, dropout=0.3): # добавим дропаут и дополнительные рекуррентные слои чтобы он работал
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind["<pad>"])
        self.rnn = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)

        self.linear = nn.Linear(hidden_dim, vocab_size)
    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        output, _ = self.rnn(embeddings)
        logits = self.linear(output)

        return logits

In [63]:
model2 = LanguageModel_2(
    vocab_size=len(word2ind),
    embedding_dim=128,
    hidden_dim=256,
    num_layers=3,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-3)

train_losses2, eval_perplexities2 = train_model(
    model=model2,
    optimizer=optimizer,
    criterion=criterion,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    num_epochs=10
)

1/10, train_loss=6.0761, eval_perplexity=277.2352
2/10, train_loss=5.2712, eval_perplexity=148.3551
3/10, train_loss=4.8674, eval_perplexity=122.6500
4/10, train_loss=4.6758, eval_perplexity=113.1728
5/10, train_loss=4.5459, eval_perplexity=108.7361
6/10, train_loss=4.4453, eval_perplexity=106.7165
7/10, train_loss=4.3636, eval_perplexity=105.8209
8/10, train_loss=4.2943, eval_perplexity=105.7558
9/10, train_loss=4.2344, eval_perplexity=106.3524
10/10, train_loss=4.1813, eval_perplexity=106.9360


In [64]:
print("Final perplexity:", eval_perplexities2[-1])

Final perplexity: 106.93598760319956


### Отчет (2 балла) (еще в самом низу таблица)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

# первый эксперимент - базовая однослойная GRU-модель.
было просмотрено, как базовая модель работает с таким типом задачи. сначала было указано 10 эпох, но модель начала переобучаться:
```
1/10, train_loss=5.3390, eval_perplexity=137.4445
2/10, train_loss=4.7102, eval_perplexity=113.8684
3/10, train_loss=4.4650, eval_perplexity=106.1942
4/10, train_loss=4.2905, eval_perplexity=103.9763
5/10, train_loss=4.1522, eval_perplexity=104.2675
6/10, train_loss=4.0370, eval_perplexity=105.7723
7/10, train_loss=3.9394, eval_perplexity=108.0792
```
поэтому было сокращено кол-во эпох до 5 и в иитоге Final perplexity: 104.36214397383723

# второй эксперимент
заключается в изменении архитектуры: использование GRU модели для улучшения запоминания контекста, что особенно важно для таких задач, добавление дропаута между несколькими рекуррентными слоями для лучшего треина модели. сначала из-за добавления дропаута и усложнения модели метрика была намного выше, чем на 1-й эпохе у базовой модели, но к концу обучение пошло лучше. Final perplexity: 106.93598760319956

### Третий эксперимент

я попробую использовать несколько линейных слоев в конце, чтобы посмотреть, даст ли более сложная голова к уменьшению метрики

In [65]:
class LanguageModel_3(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=1, dropout=0.0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind["<pad>"])
        self.rnn = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)

        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fa = nn.Tanh()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        output, _ = self.rnn(embeddings)
        output = self.fc1(output)
        output = self.fa(output)
        output = self.dropout(output)
        logits = self.fc2(output)

        return logits

In [68]:
model3 = LanguageModel_3(
    vocab_size=len(word2ind),
    embedding_dim=128,
    hidden_dim=256,
    num_layers=3,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model3.parameters(), lr=1e-3)

train_losses3, eval_perplexities3 = train_model(
    model=model3,
    optimizer=optimizer,
    criterion=criterion,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    num_epochs=9
)

1/9, train_loss=5.9748, eval_perplexity=204.2633
2/9, train_loss=5.1802, eval_perplexity=148.6083
3/9, train_loss=4.9520, eval_perplexity=128.8663
4/9, train_loss=4.8234, eval_perplexity=119.2789
5/9, train_loss=4.7355, eval_perplexity=116.7512
6/9, train_loss=4.6688, eval_perplexity=109.9745
7/9, train_loss=4.6145, eval_perplexity=108.2932
8/9, train_loss=4.5682, eval_perplexity=105.5798
9/9, train_loss=4.5299, eval_perplexity=104.3085


In [69]:
print("Final perplexity:", eval_perplexities3[-1])

Final perplexity: 104.30845887445236


# третий эксперимент (дополнительный) заключался в выявлении влияния более сложной выходной части модели (с двумя fully connected слоями) на уменьшение метрики
Final perplexity: 104.30845887445236

In [70]:
test_perplexity1 = evaluate(model1, criterion, test_dataloader)
test_perplexity2 = evaluate(model2, criterion, test_dataloader)
test_perplexity3 = evaluate(model3, criterion, test_dataloader)

In [71]:
print("model 1 test ppl:", test_perplexity1)
print("model 2 test ppl:", test_perplexity2)
print("model 3 test ppl:", test_perplexity3)

model 1 test ppl: 103.22113455331238
model 2 test ppl: 105.55974444212347
model 3 test ppl: 102.78085403899684


In [72]:
import pandas as pd

results = pd.DataFrame({
    "эксперимент": [
        "1. Базовая модель",
        "2. GRU + несколько слоев + dropout",
        "3. GRU + усложненная голова"
    ],
    "Описание": [
        "базовый вариант",
        "GRU с dropout между рекуррентными слоями, 3 слоя",
        "GRU с двумя полносвязными слоями в выходной части"
    ],
    "perplexity": [
        test_perplexity1,
        test_perplexity2,
        test_perplexity3
    ]
})

results

,эксперимент,Описание,perplexity
0,1. Базовая модель,базовый вариант,103.221135
1,2. GRU + несколько слоев + dropout,"GRU с dropout между рекуррентными слоями, 3 слоя",105.559744
2,3. GRU + усложненная голова,GRU с двумя полносвязными слоями в выходной части,102.780854


# вывод
наилучшее качество показала третья модель, для которой test perplexity составила 102.780854